# Optimize Query Performance
Mesures effect of several diffrent optimizations of the queries.



## 1. Configure Spark

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, project_root

spark = create_spark("integration-gold")
ROOT = project_root()

from src.lake import GOLD, SILVER, read_delta, show_delta, write_gold

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark:", spark.version)
print("ROOT:", ROOT)


:: loading settings :: url = jar:file:/Users/emma/PycharmProject/id2221-labs/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/emma/.ivy2/cache
The jars for the packages stored in: /Users/emma/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a2c58279-ea8a-4ace-9911-ba1e5fcf1940;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 130ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

JAVA_HOME: /opt/homebrew/opt/openjdk@17
PySpark: 3.5.4
ROOT: /Users/emma/PycharmProject/id2221-labs


## 2. Load silver tables

In [2]:
trips = read_delta(spark, SILVER / "taxi_trips")
weather = read_delta(spark, SILVER / "weather")
air_quality = read_delta(spark, SILVER / "air_quality")
zones = read_delta(spark, SILVER / "taxi_zones")

## 3. Benchmark and Verification 

In [3]:
import time

def benchmark_and_verify(df_baseline, df_optimized, name="Optimization Test"):
    """
    Measures execution time and verifies result row count integrity.
    """
    print(f"=== {name} ===")
    
    # Measure Baseline
    t0 = time.time()
    base_count = df_baseline.count()
    base_time = time.time() - t0
    
    # Measure Optimized
    t0 = time.time()
    opt_count = df_optimized.count()
    opt_time = time.time() - t0
    
    # Verify Result Integrity
    assert base_count == opt_count, f"Mismatch! Baseline: {base_count}, Optimized: {opt_count}"
    
    speedup = ((base_time - opt_time) / base_time) * 100 if base_time > 0 else 0
    print(f"Row Count:      {base_count:,}")
    print(f"Baseline Time:  {base_time:.2f}s")
    print(f"Optimized Time: {opt_time:.2f}s")
    print(f"Speedup:        {speedup:.1f}%\n")

## 3. Caching Results

In [4]:
import time
from pyspark.sql.functions import broadcast, avg

# Decrease weather to exactly 1 row per date + hour
weather_hourly = weather.groupBy("observation_date", "observation_hour") \
                        .agg(avg("temperature_c").alias("temperature_c"))

# Select minimal columns and join on date and hour to prevent row multiplication
intermediate_df = trips.filter("fare_amount > 0 AND trip_distance > 0") \
                       .select("pickup_date", "pickup_hour", "fare_amount", "trip_distance") \
                       .join(
                           broadcast(weather_hourly),
                           (trips.pickup_date == weather_hourly.observation_date) & 
                           (trips.pickup_hour == weather_hourly.observation_hour),
                           "inner"
                       )

# Baseline: uncached
t0 = time.time()
count1_base = intermediate_df.groupBy("pickup_hour").avg("fare_amount").count()
count2_base = intermediate_df.groupBy("temperature_c").avg("trip_distance").count()
base_time = time.time() - t0

# Optimized: cached
cached_df = intermediate_df.cache()
cached_df.count()

t0 = time.time()
count1_opt = cached_df.groupBy("pickup_hour").avg("fare_amount").count()
count2_opt = cached_df.groupBy("temperature_c").avg("trip_distance").count()
opt_time = time.time() - t0

# Clean up memory storage
cached_df.unpersist()

print(f"=== Cache Performance ===")
print(f"Baseline Time (Uncached, 2 queries): {base_time:.2f}s")
print(f"Optimized Time (Cached, 2 queries):  {opt_time:.2f}s")
print(f"Speedup: {((base_time - opt_time)/base_time)*100:.1f}%")

26/09/18 23:29:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


=== Cache Performance ===
Baseline Time (Uncached, 2 queries): 27.12s
Optimized Time (Cached, 2 queries):  7.04s
Speedup: 74.0%


## 4. Partition Pruning

In [5]:
# Baseline: filtering on timestamp expression
df_base = trips.filter("date(pickup_datetime) = '2025-01-01'")

# Optimized: explicitly filtering on the partition column 'pickup_date'
df_opt = trips.filter("pickup_date = '2025-01-01' AND date(pickup_datetime) = '2025-01-01'")

benchmark_and_verify(df_base, df_opt, "Partition Pruning Performance")

=== Partition Pruning Performance ===


Row Count:      83,425
Baseline Time:  2.01s
Optimized Time: 2.07s
Speedup:        -2.9%



## 5. Optimize Broadcast Joins

In [6]:
from pyspark.sql.functions import broadcast

# Baseline: standard sort-and-merge join
df_base = trips.join(zones, trips.pickup_location_id == zones.location_id)

# Optimized: broadcast small table
df_opt = trips.join(broadcast(zones), trips.pickup_location_id == zones.location_id)

benchmark_and_verify(df_base, df_opt, "Broadcast Join Performance")

=== Broadcast Join Performance ===


Row Count:      18,961,395
Baseline Time:  9.05s
Optimized Time: 3.03s
Speedup:        66.5%



## 6. Adaptive Query Execution (AQE)

In [7]:
# Query: multi-stage aggregation with shuffles
query_df = trips.groupBy("pickup_location_id", "pickup_hour") \
                .agg({"fare_amount": "avg", "trip_distance": "sum"})

# Baseline: AQE disabled
spark.conf.set("spark.sql.adaptive.enabled", "false")
t0 = time.time()
count_disabled = query_df.count()
time_disabled = time.time() - t0

# Optimized: AQE enabled
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
t0 = time.time()
count_enabled = query_df.count()
time_enabled = time.time() - t0

assert count_disabled == count_enabled, "AQE Result Mismatch!"

print(f"=== AQE Performance ===")
print(f"AQE Disabled Time: {time_disabled:.2f}s")
print(f"AQE Enabled Time:  {time_enabled:.2f}s")
print(f"Speedup:           {((time_disabled - time_enabled)/time_disabled)*100:.1f}%\n")

=== AQE Performance ===
AQE Disabled Time: 6.89s
AQE Enabled Time:  3.42s
Speedup:           50.4%

